## 고장 핫스팟 데이터 전처리 및 EDA


### 1. 기본 설정


In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

base_path = '/content/drive/MyDrive/26-1_BITAmin_TS_project'

RAW_DIR = os.path.join(base_path, 'raw_data')
USAGE_DIR = os.path.join(RAW_DIR, 'usage')
REPAIR_DIR = os.path.join(RAW_DIR, 'repair')
PROCESSED_DIR = os.path.join(base_path, 'processed_data')
os.makedirs(PROCESSED_DIR, exist_ok=True)

usage_files = sorted(glob.glob(os.path.join(USAGE_DIR, '*.csv')))
repair_files = sorted(glob.glob(os.path.join(REPAIR_DIR, '*.csv')))

print('base_path:', base_path)
print('usage 파일 수:', len(usage_files))
print('repair 파일 수:', len(repair_files))
print('processed_dir:', PROCESSED_DIR)
print('usage 예시:', os.path.basename(usage_files[0]) if usage_files else '없음')
print('repair 예시:', os.path.basename(repair_files[0]) if repair_files else '없음')

if len(usage_files) == 0 or len(repair_files) == 0:
    print('FAIL: raw_data/usage 또는 raw_data/repair 파일 확인 필요')
else:
    print('PASS: 입력 파일 확인 완료')



Mounted at /content/drive
usage 파일 수: 59
repair 파일 수: 12


### 2. 공통 함수 설정


In [3]:
def read_csv_with_fallback(path):
    for enc in ['cp949', 'utf-8-sig', 'utf-8']:
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False)
        except UnicodeDecodeError:
            continue
    raise ValueError(f'인코딩 확인 필요: {path}')


def normalize_bike_id(s):
    s = s.astype('string').str.strip().str.upper()
    s = s.replace({'': pd.NA, 'NAN': pd.NA, 'NONE': pd.NA})
    return s


def normalize_station_id(s):
    s = s.astype('string').str.strip()
    s = s.str.replace(r'\.0$', '', regex=True)
    s = s.str.replace(r'[^0-9]', '', regex=True)
    s = s.replace({'': pd.NA})
    return s


def read_location_file(path):
    ext = os.path.splitext(path)[1].lower()
    if ext in ['.xlsx', '.xls']:
        return pd.read_excel(path)
    return read_csv_with_fallback(path)



### 3. usage 데이터 전처리


In [ ]:
usage_cache = os.path.join(PROCESSED_DIR, 'usage_core.parquet')
usage_need = ['자전거번호', '대여일시', '반납일시', '대여 대여소번호', '반납대여소번호']

if os.path.exists(usage_cache):
    usage_core = pd.read_parquet(usage_cache)
    usage_core = usage_core[[c for c in usage_need if c in usage_core.columns]].copy()
    print('기존 parquet 로드:', usage_cache)
else:
    usage_chunks = []
    stats = []

    for fp in usage_files:
        df = read_csv_with_fallback(fp).copy()
        df.columns = df.columns.str.strip()

        rename_map = {
            '대여대여소번호': '대여 대여소번호',
            '대여 대여소 번호': '대여 대여소번호',
            '반납 대여소번호': '반납대여소번호',
            '반납 대여소 번호': '반납대여소번호',
        }
        df = df.rename(columns=rename_map)

        for c in usage_need:
            if c not in df.columns:
                df[c] = pd.NA

        cut = df[usage_need].copy()

        cut['자전거번호'] = normalize_bike_id(cut['자전거번호'])
        cut['대여 대여소번호'] = normalize_station_id(cut['대여 대여소번호'])
        cut['반납대여소번호'] = normalize_station_id(cut['반납대여소번호'])
        cut['대여일시'] = pd.to_datetime(cut['대여일시'], errors='coerce')
        cut['반납일시'] = pd.to_datetime(cut['반납일시'], errors='coerce')

        valid = (
            cut['자전거번호'].notna()
            & cut['대여일시'].notna()
            & cut['반납일시'].notna()
            & (cut['반납일시'] >= cut['대여일시'])
            & cut['반납대여소번호'].notna()
        )

        dropped = int((~valid).sum())
        usage_chunks.append(cut.loc[valid].copy())
        stats.append({'file': os.path.basename(fp), 'total': len(cut), 'dropped': dropped})

    usage_core = pd.concat(usage_chunks, ignore_index=True)

    stats_df = pd.DataFrame(stats)
    print('원본 건수:', int(stats_df['total'].sum()))
    print('제거 건수:', int(stats_df['dropped'].sum()))

usage_core = usage_core[usage_need].copy()
usage_core.to_parquet(usage_cache, index=False)

print('usage_core shape:', usage_core.shape)
print('usage_core columns:', usage_core.columns.tolist())
print('결측(핵심):')
print(usage_core[['자전거번호','대여일시','반납일시','반납대여소번호']].isna().sum())
print('시간 역전 건수:', int((usage_core['반납일시'] < usage_core['대여일시']).sum()))
usage_core.head()



### 4. repair 데이터 전처리


In [10]:
repair_cache = os.path.join(PROCESSED_DIR, 'repair_core.parquet')
repair_need = ['자전거번호', '등록일시', '고장구분']

if os.path.exists(repair_cache):
    repair_core = pd.read_parquet(repair_cache)
    repair_core = repair_core[[c for c in repair_need if c in repair_core.columns]].copy()
    print('기존 parquet 로드:', repair_cache)
else:
    repair_chunks = []
    stats = []

    for fp in repair_files:
        df = read_csv_with_fallback(fp).copy()
        df.columns = df.columns.str.strip()

        if '구분' in df.columns and '고장구분' not in df.columns:
            df = df.rename(columns={'구분': '고장구분'})

        for c in repair_need:
            if c not in df.columns:
                df[c] = pd.NA

        cut = df[repair_need].copy()
        cut['자전거번호'] = normalize_bike_id(cut['자전거번호'])
        cut['고장구분'] = cut['고장구분'].astype('string').str.strip()

        reg = cut['등록일시'].astype('string').str.strip()
        reg = reg.str.replace('.', '-', regex=False).str.replace('/', '-', regex=False)
        reg = reg.replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA})
        cut['등록일시'] = pd.to_datetime(reg, errors='coerce')

        valid = cut['자전거번호'].notna() & cut['등록일시'].notna()
        dropped = int((~valid).sum())

        repair_chunks.append(cut.loc[valid].copy())
        stats.append({'file': os.path.basename(fp), 'total': len(cut), 'dropped': dropped})

    repair_core = pd.concat(repair_chunks, ignore_index=True)

    stats_df = pd.DataFrame(stats)
    print('원본 건수:', int(stats_df['total'].sum()))
    print('제거 건수:', int(stats_df['dropped'].sum()))

repair_core = repair_core[repair_need].copy()
repair_core = repair_core.sort_values(['등록일시']).reset_index(drop=True)
repair_core.to_parquet(repair_cache, index=False)

print('repair_core shape:', repair_core.shape)
print('repair_core columns:', repair_core.columns.tolist())
print('결측(핵심):')
print(repair_core[['자전거번호','등록일시']].isna().sum())
repair_core.head()



shape: (779052, 3)
columns: ['자전거번호', '등록일시', '고장구분']


,자전거번호,등록일시,고장구분
0,SPB-41936,2021-01-01 00:44:00,기타
1,SPB-42181,2021-01-01 02:42:00,타이어
2,SPB-36237,2021-01-01 03:53:00,기타
3,SPB-33399,2021-01-01 04:56:00,체인
4,SPB-36328,2021-01-01 08:21:00,기타
...,...,...,...
779047,SPB-68754,2025-12-31 23:12:55,체인
779048,SPB-62680,2025-12-31 23:32:06,기타
779049,SPB-34168,2025-12-31 23:33:45,체인
779050,SPB-34168,2025-12-31 23:33:45,기타


### 5. usage + repair 데이터 매칭 (자전거번호 기준)


In [ ]:
usage_match = usage_core[['자전거번호', '반납일시', '반납대여소번호', '대여 대여소번호']].copy()
usage_match = usage_match.sort_values(['자전거번호', '반납일시']).reset_index(drop=True)

repair_match = repair_core[['자전거번호', '등록일시', '고장구분']].copy()
repair_match = repair_match.sort_values(['자전거번호', '등록일시']).reset_index(drop=True)

usage_bikes = set(usage_match['자전거번호'].dropna().unique())
repair_bikes = set(repair_match['자전거번호'].dropna().unique())
common_bikes = usage_bikes & repair_bikes
print('usage 자전거 수:', len(usage_bikes))
print('repair 자전거 수:', len(repair_bikes))
print('공통 자전거 수:', len(common_bikes))

repair_station_events = pd.merge_asof(
    repair_match,
    usage_match,
    by='자전거번호',
    left_on='등록일시',
    right_on='반납일시',
    direction='backward',
    allow_exact_matches=True,
)

repair_station_events = repair_station_events.rename(columns={'반납대여소번호': '추정고장대여소ID'})
repair_station_events['시간차_분'] = (
    repair_station_events['등록일시'] - repair_station_events['반납일시']
).dt.total_seconds() / 60
repair_station_events['위치매핑성공'] = repair_station_events['추정고장대여소ID'].notna()

repair_station_events_path = os.path.join(PROCESSED_DIR, 'repair_station_events.parquet')
repair_station_events.to_parquet(repair_station_events_path, index=False)

print('저장:', repair_station_events_path)
print('총 고장 이벤트:', len(repair_station_events))
print('위치 매핑 성공률(%):', round(repair_station_events['위치매핑성공'].mean() * 100, 4))
repair_station_events.head()



In [ ]:
# 병합 데이터 검증
print('[병합 데이터 검증]')
same_len = len(repair_station_events) == len(repair_core)
neg_gap = int((repair_station_events['시간차_분'] < 0).fillna(False).sum())
req = ['자전거번호','등록일시','반납일시','추정고장대여소ID','위치매핑성공']
missing = [c for c in req if c not in repair_station_events.columns]

print('행 개수 일치:', same_len)
print('필수 컬럼 누락:', missing)
print('음수 시간차:', neg_gap)

if same_len and neg_gap == 0 and len(missing) == 0:
    print('PASS')
else:
    print('FAIL')



### 6. 고장 핫스팟 집계
집계 A: 대여소별 총 고장건수
<br> 집계 B: 대여소-일자별 고장건수 (빈도 추이)


In [ ]:
fault_base = repair_station_events[repair_station_events['위치매핑성공']].copy()
fault_base['date'] = fault_base['등록일시'].dt.floor('D')

station_hotspot = (
    fault_base
    .groupby('추정고장대여소ID', as_index=False)
    .agg(
        고장건수=('등록일시', 'count'),
        고장자전거수=('자전거번호', 'nunique'),
        고장유형수=('고장구분', 'nunique'),
        첫고장일=('등록일시', 'min'),
        마지막고장일=('등록일시', 'max')
    )
    .sort_values('고장건수', ascending=False)
    .reset_index(drop=True)
)

station_hotspot_daily = (
    fault_base
    .groupby(['date', '추정고장대여소ID'], as_index=False)
    .agg(고장건수=('등록일시', 'count'))
    .sort_values(['date', '고장건수'], ascending=[True, False])
    .reset_index(drop=True)
)

hotspot_path = os.path.join(PROCESSED_DIR, 'station_hotspot.parquet')
hotspot_daily_path = os.path.join(PROCESSED_DIR, 'station_hotspot_daily.parquet')
station_hotspot.to_parquet(hotspot_path, index=False)
station_hotspot_daily.to_parquet(hotspot_daily_path, index=False)

print('저장:', hotspot_path)
print('저장:', hotspot_daily_path)
print('station_hotspot shape:', station_hotspot.shape)
print('station_hotspot_daily shape:', station_hotspot_daily.shape)
station_hotspot.head(10)



In [ ]:
# 핫스팟 검증
print('[STEP 5 검증]')
print('핫스팟 대여소 수:', station_hotspot['추정고장대여소ID'].nunique())
print('총 고장건수 합:', int(station_hotspot['고장건수'].sum()))
print('daily 총 고장건수 합:', int(station_hotspot_daily['고장건수'].sum()))

f1 = os.path.join(PROCESSED_DIR, 'station_hotspot.parquet')
f2 = os.path.join(PROCESSED_DIR, 'station_hotspot_daily.parquet')
print('file1 exists/size:', os.path.exists(f1), os.path.getsize(f1) if os.path.exists(f1) else 0)
print('file2 exists/size:', os.path.exists(f2), os.path.getsize(f2) if os.path.exists(f2) else 0)

if int(station_hotspot['고장건수'].sum()) == int(station_hotspot_daily['고장건수'].sum()):
    print('PASS')
else:
    print('FAIL: 집계 합계 불일치')



### 7. EDA 핫스팟 시각화


In [ ]:
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.unicode_minus'] = False

# 한글 폰트
try:
    import matplotlib.font_manager as fm
    import os
    os.system("apt-get -qq install fonts-nanum > /dev/null")
    plt.rc('font', family='NanumGothic')
except:
    pass

top20 = station_hotspot.head(20)
fig, ax = plt.subplots()
ax.bar(top20['추정고장대여소ID'].astype(str), top20['고장건수'])
ax.set_title('고장 핫스팟 TOP 20 (대여소별 고장건수)')
ax.set_xlabel('대여소ID')
ax.set_ylabel('고장건수')
ax.tick_params(axis='x', rotation=90)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()



In [ ]:
# 월별 전체 고장 추이
monthly = station_hotspot_daily.copy()
monthly['year_month'] = pd.to_datetime(monthly['date']).dt.to_period('M').astype(str)
monthly_sum = monthly.groupby('year_month', as_index=False)['고장건수'].sum()

fig, ax = plt.subplots()
ax.plot(monthly_sum['year_month'], monthly_sum['고장건수'], marker='o')
ax.set_title('월별 고장 발생 건수')
ax.set_xlabel('year_month')
ax.set_ylabel('고장건수')
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### 8. 위치 좌표 조인 후 지도용 parquet 작업


In [ ]:
location_candidates = []
location_candidates += sorted(glob.glob(os.path.join(RAW_DIR, '*위치*.csv')))
location_candidates += sorted(glob.glob(os.path.join(RAW_DIR, '*대여소*.csv')))
location_candidates += sorted(glob.glob(os.path.join(RAW_DIR, '*station*.csv')))
location_candidates += sorted(glob.glob(os.path.join(RAW_DIR, '*위치*.xlsx')))
location_candidates += sorted(glob.glob(os.path.join(RAW_DIR, '*대여소*.xlsx')))
location_candidates += sorted(glob.glob(os.path.join(RAW_DIR, '*station*.xlsx')))
location_candidates = sorted(set(location_candidates))

print('위치 파일 후보:', [os.path.basename(x) for x in location_candidates])

if location_candidates:
    loc_path = location_candidates[0]
    loc = read_location_file(loc_path)
    loc.columns = loc.columns.str.strip()

    id_candidates = ['대여소ID', '대여소번호', '대여소 번호', '대여소_ID', 'station_id', 'STATION_ID']
    lat_candidates = ['위도', 'lat', 'LAT', 'latitude', 'Latitude']
    lon_candidates = ['경도', 'lon', 'LON', 'lng', 'longitude', 'Longitude']

    id_col = next((c for c in id_candidates if c in loc.columns), None)
    lat_col = next((c for c in lat_candidates if c in loc.columns), None)
    lon_col = next((c for c in lon_candidates if c in loc.columns), None)

    if id_col and lat_col and lon_col:
        station_location_core = loc[[id_col, lat_col, lon_col]].copy()
        station_location_core = station_location_core.rename(columns={id_col:'추정고장대여소ID', lat_col:'위도', lon_col:'경도'})

        station_location_core['추정고장대여소ID'] = normalize_station_id(station_location_core['추정고장대여소ID'])
        station_location_core['위도'] = pd.to_numeric(station_location_core['위도'], errors='coerce')
        station_location_core['경도'] = pd.to_numeric(station_location_core['경도'], errors='coerce')

        valid_coord = station_location_core['위도'].between(33, 39) & station_location_core['경도'].between(124, 132)
        station_location_core = station_location_core[valid_coord]
        station_location_core = station_location_core.dropna(subset=['추정고장대여소ID','위도','경도']).drop_duplicates('추정고장대여소ID').reset_index(drop=True)

        loc_core_path = os.path.join(PROCESSED_DIR, 'station_location_core.parquet')
        station_location_core.to_parquet(loc_core_path, index=False)

        station_hotspot_map = station_hotspot.merge(station_location_core, on='추정고장대여소ID', how='left')
        station_hotspot_map['좌표매핑성공'] = station_hotspot_map['위도'].notna() & station_hotspot_map['경도'].notna()

        out_path = os.path.join(PROCESSED_DIR, 'station_hotspot_with_coords.parquet')
        station_hotspot_map.to_parquet(out_path, index=False)

        print('사용 위치 파일:', os.path.basename(loc_path))
        print('location core 저장:', loc_core_path)
        print('hotspot+coords 저장:', out_path)
        print('좌표 매핑률(%):', round(station_hotspot_map['좌표매핑성공'].mean() * 100, 4))
        display(station_location_core.head())
        display(station_hotspot_map.head())
    else:
        print('위치 파일에서 대여소ID/위도/경도 컬럼을 찾지 못했습니다.')
        print('현재 컬럼:', loc.columns.tolist())
else:
    print('위치 파일이 없어서 지도용 조인은 생략합니다.')

